In this notebook, we prep the catalog to be used!!

In [81]:
import os
import sys
import glob
import numpy as np
from astropy.io import fits
from astropy.table import Table, vstack, hstack
import pandas as pd

rootdir = '/global/u1/v/virajvm/'
sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code'))

%load_ext autoreload
%autoreload 2



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [82]:
filename = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dr1_dwarf_catalog.fits"

# load the MAIN extension directly as an Astropy Table
data_cat = Table.read(filename, hdu="MAIN")
fspec_cat = Table.read(filename, hdu="FASTSPEC")
spec_cat = Table.read(filename, hdu="SPECTRA_TEMPLATE")

mask = (data_cat["DWARF_MASKBIT"] == 0)

data_cat = data_cat[mask]
fspec_cat = fspec_cat[mask]
spec_cat = spec_cat[mask]



In [83]:
main_cat = data_cat["TARGETID","Z","RA","DEC","LOG_MSTAR_M24", "MAG_R","SAMPLE"]



spec_cat = spec_cat["TARGETID","SPEC_UMAP_0", "SPEC_UMAP_1","NNMF_RESID"]

tot_cat = hstack([main_cat, fspec_cat["HALPHA_FLUX"]])
ha_mask = fspec_cat["HALPHA_FLUX"].data.mask
tot_cat["HALPHA_FLUX"][ha_mask] = 0

spec_cat = hstack([spec_cat, main_cat["SAMPLE"]])

spec_cat = spec_cat[spec_cat["SPEC_UMAP_0"] > -50 ]


In [75]:
# Convert all columns except TARGETID and SAMPLE to float16
for col in tot_cat.colnames:
    if col not in ["TARGETID", "SAMPLE"]:
        if col in ["RA","DEC"]:
            tot_cat[col] = tot_cat[col].astype('float32')
        else:
            tot_cat[col] = tot_cat[col].astype('float16')

# Compute total size in bytes
total_bytes = sum(tot_cat[col].nbytes for col in tot_cat.colnames)
total_mb = total_bytes / (1024**2)
print(f"Total catalog size: {total_mb:.2f} MB")

# Convert to Pandas DataFrame and save as CSV
df = tot_cat.to_pandas()
df.to_csv("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/desi_dwarfs.csv", index=False)


Total catalog size: 11.37 MB


In [84]:
spec_cat['SPEC_UMAP_0'] = spec_cat['SPEC_UMAP_0'].astype(np.float32)
spec_cat['SPEC_UMAP_1'] = spec_cat['SPEC_UMAP_1'].astype(np.float32)

#  Convert NNMF_RESID to float16
spec_cat['NNMF_RESID'] = spec_cat['NNMF_RESID'].astype(np.float16)

# Convert SAMPLE string column to 4 binary columns
samples = ['ELG', 'BGS_BRIGHT', 'BGS_FAINT', 'LOWZ']

# Initialize new columns with zeros
for s in samples:
    col_name = f'in_{s.replace("BGS_BRIGHT","BGSB").replace("BGS_FAINT","BGSF")}'
    spec_cat[col_name] = np.zeros(len(spec_cat), dtype=np.int8)  # 0/1 column

# Fill in ones according to SAMPLE
for i, row in enumerate(spec_cat):
    if row['SAMPLE'] == 'ELG':
        spec_cat['in_ELG'][i] = 1
    elif row['SAMPLE'] == 'BGS_BRIGHT':
        spec_cat['in_BGSB'][i] = 1
    elif row['SAMPLE'] == 'BGS_FAINT':
        spec_cat['in_BGSF'][i] = 1
    elif row['SAMPLE'] == 'LOWZ':
        spec_cat['in_LOWZ'][i] = 1

# Remove the original SAMPLE column if you like
spec_cat.remove_column('SAMPLE')



In [85]:
df_spec = spec_cat.to_pandas()
df_spec.to_csv("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/umap_catalog.csv", index=False)


In [86]:
df_spec

,TARGETID,SPEC_UMAP_0,SPEC_UMAP_1,NNMF_RESID,in_ELG,in_BGSB,in_BGSF,in_LOWZ
0,39627066986994125,12.842730,4.534079,51.56250,0,1,0,0
1,39627067007963313,12.498918,2.934067,53.62500,0,1,0,0
2,39627072187928583,6.793174,4.161978,54.90625,0,1,0,0
3,39627072192121852,10.536270,-0.548725,52.81250,0,1,0,0
4,39627077380476509,10.724737,0.587644,52.96875,0,1,0,0
...,...,...,...,...,...,...,...,...
350623,39627636598641999,11.761548,1.646974,57.65625,0,1,0,0
350624,39627769058951413,8.642703,9.014146,80.18750,0,1,0,0
350625,39627351838953626,9.755444,2.125329,57.84375,0,1,0,0
350626,39627764344553699,10.637001,1.671248,66.00000,0,1,0,0
